# Module 5: uncertainty and risk, seasonal analysis

This notebook follows the fifth day of a DSSAT training course. A single season says little about a
management choice: the next season's weather is unknown. Each exercise here runs a choice over many seasons and
judges it by the whole distribution of outcomes.

1. **Seasonal analysis**: the irrigation strategies of `UFGA7812.SNX` over 30 seasons of generated weather, with
   the experiment's price and cost file.
2. **The nitrogen exercise**: the economically best urea rate for maize at Kadoma, Zimbabwe, built as
   `KADO9901.SNX` exactly as the course specifies, for each of 13 seasons and as one "blanket" rate.
3. **Regional risk analysis and adoption**: two millet varieties on two soils at Tillabery, Niger
   (`NIGE1101.SNX`), a price penalty for late maturity, and the variety a risk-averse farmer would choose.

**Data.** `Seasonal/UFGA7812.SNX`, `Economic/UFGA7812.PRI`, `Weather/Climate/UFGA.CLI`, the Kadoma and Tillabery
weather files and the soil file from [DSSAT/dssat-csm-data](https://github.com/DSSAT/dssat-csm-data) (BSD-3);
`NIGE1101.SNX` from the course data (`notebooks/course-data/`).

In [1]:
import sys, time, subprocess, re
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, str(Path.cwd()))      # docs/notebooks
import learnlib as L
from wth import read_wth

M = "module5"
print("engine   :", L.config.engine_exe())
print("csm-data :", L.CSM_DATA, "@", L.CSM_DATA_COMMIT[:8])

engine   : C:\TEMP\dsn\dssat\dscsm048.exe
csm-data : C:\Users\benny\OneDrive\Documents\Github\dssat-csm-data @ 99c95b34


## 1. Seasonal analysis: irrigation strategies over 30 seasons

`UFGA7812.SNX` runs Bragg soybean for 30 seasons from 1978. Treatment 1 is rainfed; treatments 2 to 10
irrigate automatically when the available soil water in the top 30 cm falls to 10%, 20% ... 90% of its
capacity. The weather option `WTHER` is `W`: the engine generates daily weather from the monthly statistics
of the climate file `UFGA.CLI` (the WGEN generator), so the 30 seasons are 30 samples of Gainesville's
climate, not the weather of 1978-2007.

In [2]:
d = L.work_dir("m5_seasonal")
L.stage(d, "Seasonal/UFGA7812.SNX", "Weather/Climate/UFGA.CLI", "Soil/SOIL.SOL", "Economic/UFGA7812.PRI")
txt = (d / "UFGA7812.SNX").read_text(encoding="latin-1")
print(txt[txt.index("*SIMULATION CONTROLS"):][:700])
s, _, r = L.run(d, "UFGA7812.SNX", outputs="summary")
print(f"{len(s)} seasons in {r.seconds:.0f} s")
names = {1: "rainfed", **{k: f"auto {10 * (k - 1)}%" for k in range(2, 11)}}
seas = s[["TRNO", "SDAT", "PDAT", "MDAT", "HWAM", "IRCM", "IR#M", "PRCM", "ETCM", "TMAXA", "TMINA", "SRADA"]].copy()
seas = seas.apply(pd.to_numeric, errors="coerce")
seas["season"] = (seas.SDAT // 1000).astype(int)
seas["treatment"] = seas.TRNO.map(names)
seas.groupby("treatment", sort=False)[["HWAM", "IRCM", "IR#M", "PRCM"]].agg(["mean", "std"]).round(0)

*SIMULATION CONTROLS
@N GENERAL     NYERS NREPS START SDATE RSEED SNAME.................... SMODEL
 1 GE             30     1     S 78166  9875 Rainfed
@N OPTIONS     WATER NITRO SYMBI PHOSP POTAS DISES  CHEM  TILL   CO2
 1 OP              Y     Y     Y     N     N     N     N     Y     D
@N METHODS     WTHER INCON LIGHT EVAPO INFIL PHOTO HYDRO NSWIT MESOM MESEV MESOL
 1 ME              W     M     E     R     S     L     R     1     G     R     2
@N MANAGEMENT  PLANT IRRIG FERTI RESID HARVS
 1 MA              R     N     R     N     M
@N OUTPUTS     FNAME OVVEW SUMRY FROPT GROUT CAOUT WAOUT NIOUT MIOUT DIOUT VBOSE CHOUT OPOUT FMOPT
 1 OU              Y     N     Y     1     N     N     N   


300 seasons in 15 s


HWAM          IRCM        IR#M        PRCM       
             mean    std   mean   std  mean  std   mean    std
treatment                                                     
rainfed    2932.0  537.0    0.0   0.0   0.0  0.0  663.0  180.0
auto 10%   2987.0  467.0    8.0  12.0   0.0  1.0  667.0  179.0
auto 20%   3267.0  321.0   32.0  20.0   2.0  1.0  684.0  178.0
auto 30%   3489.0  224.0   72.0  41.0   4.0  2.0  651.0  161.0
auto 40%   3688.0  128.0  106.0  46.0   6.0  3.0  640.0  163.0
auto 50%   3780.0  104.0  144.0  47.0   9.0  3.0  637.0  161.0
auto 60%   3820.0  107.0  175.0  46.0  13.0  3.0  631.0  161.0
auto 70%   3840.0  105.0  203.0  48.0  17.0  4.0  631.0  161.0
auto 80%   3855.0  103.0  240.0  53.0  24.0  5.0  633.0  163.0
auto 90%   3854.0  103.0  295.0  51.0  35.0  6.0  633.0  163.0

### From yields to money

The price file `UFGA7812.PRI` gives, per treatment, the soybean price as a normal distribution (mean and
standard deviation, `IDIS` = 3) and fixed costs. Their meaning is in the engine's `ECONOMIC.CDE`: `GRAN`
grain price in $/t, `BASE` base production costs in $/ha, `IRRI` irrigation cost in $/mm, `IRCO` cost per
irrigation application in $/ha. The gross margin of a season is

`price x yield - BASE - IRRI x irrigation - IRCO x number of irrigations`.

Seed costs (`SCOS`, $/kg) are left out: the file gives no seeding rate in kg. One price is drawn per season
and used for all ten treatments, so the treatments are compared under the same prices.

In [3]:
pri = (d / "UFGA7812.PRI").read_text(encoding="latin-1").splitlines()
blocks, cur = {}, None
for i, ln in enumerate(pri):
    m = re.match(r"\* TREATMENT\s+(\d+)", ln)
    if m:
        cur = int(m.group(1))
    if ln.startswith("@PRAM") and cur:
        hdr = ln.split()[1:]
        tab = {}
        for k in range(1, 5):
            parts = pri[i + k].split()
            tab[parts[0]] = dict(zip(hdr, map(float, parts[1:])))
        blocks[cur] = pd.DataFrame(tab)
prices = pd.concat({k: v for k, v in blocks.items()}, names=["TRNO", "item"]).reset_index()
p1 = blocks[1]
print(p1.loc[["GRAN", "BASE", "IRRI", "IRCO", "SCOS"]])
same = all(blocks[k].equals(blocks[1]) for k in blocks)
print("all treatments have the same prices and costs:", same)

rng = np.random.default_rng(78)
years = sorted(seas.season.unique())
price = dict(zip(years, rng.normal(p1.loc["GRAN", "PAR1"], p1.loc["GRAN", "PAR2"], len(years))))
seas["price"] = seas.season.map(price)
seas["margin"] = (seas.price * seas.HWAM / 1000 - p1.loc["BASE", "PAR1"]
                  - p1.loc["IRRI", "PAR1"] * seas.IRCM - p1.loc["IRCO", "PAR1"] * seas["IR#M"])
risk = seas.groupby("treatment", sort=False).agg(mean_yield=("HWAM", "mean"), sd_yield=("HWAM", "std"),
                                                 mean_irr=("IRCM", "mean"), mean_margin=("margin", "mean"),
                                                 sd_margin=("margin", "std"), p10_margin=("margin", lambda m: m.quantile(0.1)),
                                                 min_margin=("margin", "min")).reset_index()
risk

      IDIS    PAR1  PAR2  PAR3
GRAN   3.0  320.00  32.0   0.0
BASE   0.0  120.00   0.0   0.0
IRRI   0.0    0.50   0.0   0.0
IRCO   0.0    0.50   0.0   0.0
SCOS   0.0    0.46   0.0   0.0
all treatments have the same prices and costs: True


,treatment,mean_yield,sd_yield,mean_irr,mean_margin,sd_margin,p10_margin,min_margin
0,rainfed,2932.233333,537.405689,0.000000,860.165100,213.526974,587.408884,495.871313
1,auto 10%,2986.900000,467.080836,8.066667,874.048685,197.324293,637.420581,585.808572
2,auto 20%,3267.433333,321.003350,32.466667,952.684211,149.660359,773.998045,690.161076
3,auto 30%,3489.133333,223.695257,71.533333,1005.287026,134.030972,854.405328,784.540699
4,auto 40%,3688.166667,127.876310,105.500000,1053.276489,117.554370,918.755106,889.340005
5,auto 50%,3779.966667,104.085669,143.966667,1062.526753,107.818962,930.276065,914.595194
6,auto 60%,3820.066667,106.792074,175.300000,1057.809445,94.459694,933.088386,919.392102
7,auto 70%,3839.866667,104.912684,202.566667,1048.825380,95.140914,916.024867,892.085503
8,auto 80%,3855.433333,102.993700,240.200000,1031.658207,98.707458,902.085533,884.375606
9,auto 90%,3853.800000,102.661614,294.600000,998.579042,97.473282,871.182682,853.959999


### Comparing distributions

Two ways to compare strategies over the 30 seasons:

- **First-degree stochastic dominance**: strategy A dominates B if, for every margin level, A is at least as
  likely as B to reach it (A's cumulative distribution lies nowhere above B's).
- **Risk attitude**: with an exponential utility `u(m) = -exp(-r m)` the certainty equivalent
  `CE = -ln(mean(exp(-r m))) / r` is the sure margin a farmer with risk aversion `r` would accept instead of
  the uncertain one. `r = 0` is the plain mean.

In [4]:
def cdf_dominates(a, b, grid):
    Fa = np.searchsorted(np.sort(a), grid, side="right") / len(a)
    Fb = np.searchsorted(np.sort(b), grid, side="right") / len(b)
    return bool(np.all(Fa <= Fb + 1e-12) and np.any(Fa < Fb - 1e-12))

order = list(names.values())
mg = {t: seas[seas.treatment == t].margin.to_numpy() for t in order}
grid = np.sort(seas.margin.unique())
dom = pd.DataFrame([[cdf_dominates(mg[a], mg[b], grid) for b in order] for a in order], index=order, columns=order)
not_dominated = [t for t in order if not dom[t].any()]
print("not dominated by any other strategy:", not_dominated)

def ce(m, r):
    return float(m.mean()) if r == 0 else float(-np.log(np.mean(np.exp(-r * (m - m.mean())))) / r + m.mean())

rs = [0, 0.002, 0.005, 0.01, 0.02, 0.05]
ces = pd.DataFrame([{"r": r, "treatment": t, "CE": ce(mg[t], r)} for r in rs for t in order])
ces.loc[ces.groupby("r").CE.idxmax()]

not dominated by any other strategy: ['rainfed', 'auto 10%', 'auto 30%', 'auto 40%', 'auto 50%', 'auto 60%', 'auto 70%', 'auto 80%']


,r,treatment,CE
5,0.000,auto 50%,1062.526753
15,0.002,auto 50%,1051.896205
25,0.005,auto 50%,1037.983959
36,0.010,auto 60%,1020.550735
46,0.020,auto 60%,995.333470
56,0.050,auto 60%,960.781838


In [5]:
L.export(seas[["treatment", "TRNO", "season", "HWAM", "IRCM", "IR#M", "PRCM", "ETCM", "price", "margin"]], M, "seasonal")
L.export(risk, M, "seasonal_risk")
L.export(dom.reset_index().rename(columns={"index": "row"}), M, "seasonal_dominance")
L.export(prices, M, "seasonal_prices")
cli = (L.CSM_DATA / "Weather/Climate/UFGA.CLI").read_text(encoding="latin-1").splitlines()
i = next(k for k, l in enumerate(cli) if l.startswith("@  MTH  SAMN"))
clim = pd.DataFrame([l.split() for l in cli[i + 1:i + 13]], columns=cli[i].split()[1:]).apply(pd.to_numeric, errors="coerce")
L.export(clim, M, "ufga_climate")

WindowsPath('C:/Users/benny/OneDrive/Documents/Github/xdssat/docs/learn/data/module5/ufga_climate.csv')

## 2. The nitrogen exercise: maize at Kadoma, Zimbabwe

The course asks for the urea rate that maximizes the gross margin of rainfed maize at the Cotton Research
Institute, Kadoma (Natural Region II of central Zimbabwe), first for one season, then as a "blanket" rate over
all 13 seasons of measured weather (1977/78 to 1989/90). The experiment, `KADO9901.SNX`, is hypothetical and
fully specified by the exercise:

- cultivar MH-16 from Malawi, entered in the cultivar file as given: `IB0098 MH-16 . IB0001 260.0 0.600 840.0 600.0 6.00 38.90`;
- the Millhopper fine sand `IBMZ910014`, measured weather of station `KADO`;
- initial conditions on 7 October: previous crop maize, 10 kg/ha of roots, no nodules, 100 kg/ha of surface
  residue with 0.5% N, soil water at 50% of the available range, 5 kg/ha of soil N;
- planting on 15 October, 3.7 plants/m² planted and emerged, 61 cm rows, 7 cm deep;
- urea in two equal splits, at planting and 30 days later, incorporated to 5 cm; no irrigation;
- simulation from 7 October, 1 year per season (part A) or 13 years (part B).

The exercise's "5 kg/ha of soil N" is spread here as nitrate at the same concentration in every layer (no
ammonium), which is a choice: the exercise does not say how XBuild divides it.

In [6]:
from sol import read_sol
mill = read_sol(L.CSM_DATA / "Soil/SOIL.SOL", ids=["IBMZ910014"])["IBMZ910014"]
lay = mill.layers.copy()
lay["top"] = np.r_[0, lay.SLB.values[:-1]]
lay["thick"] = lay.SLB - lay.top
# kg N/ha in a layer = ppm x bulk density (g/cm3) x thickness (cm) x 0.1
ppm = 5.0 / float((lay.SBDM * lay.thick * 0.1).sum())
lay["SH2O"] = lay.SLLL + 0.5 * (lay.SDUL - lay.SLLL)
print(f"soil NO3 {ppm:.3f} ppm in every layer gives 5 kg N/ha")
rates = list(range(0, 241, 20))

def kado_filex(nyers=13, year=77):
    trt = "\n".join(f"{i:>2} 1 0 0 {f'UREA {r} KG N/HA':<25}{1:>3}{1:>3}{0:>3}{1:>3}{1:>3}{0:>3}{(i - 1 if r else 0):>3}{0:>3}{0:>3}{0:>3}{0:>3}{0:>3}{1:>3}"
                    for i, r in enumerate(rates, 1))
    ic = "\n".join(f" 1 {int(b):>5} {w:>5.3f} {0:>5} {ppm:>5.2f}" for b, w in zip(lay.SLB, lay.SH2O))
    fert = "\n".join(f"{lvl:>2} {year:02d}{doy} FE005 AP002     5 {r / 2:>5.1f}     0     0     0     0   -99 -99"
                     for lvl, r in enumerate([r for r in rates if r], 1) for doy in (288, 318))
    lines = [
        f'*EXP.DETAILS: KADO{year:02d}01SN KADOMA MAIZE N RATES (COURSE EXERCISE)',
        f'',
        f'*GENERAL',
        f'@PEOPLE',
        f'DSSAT training exercise, built in Python',
        f'@SITE',
        f'Cotton Research Institute, Kadoma, Zimbabwe',
        f'',
        f'*TREATMENTS                        -------------FACTOR LEVELS------------',
        f'@N R O C TNAME.................... CU FL SA IC MP MI MF MR MC MT ME MH SM',
        f'{trt}',
        f'',
        f'*CULTIVARS',
        f'@C CR INGENO CNAME',
        f' 1 MZ IB0098 MH-16',
        f'',
        f'*FIELDS',
        f'@L ID_FIELD WSTA....  FLSA  FLOB  FLDT  FLDD  FLDS  FLST SLTX  SLDP  ID_SOIL    FLNAME',
        f' 1 KADO0001 KADO       -99     0 DR000     0     0 00000 -99    180  IBMZ910014 -99',
        f'',
        f'*INITIAL CONDITIONS',
        f'@C   PCR ICDAT  ICRT  ICND  ICRN  ICRE  ICWD ICRES ICREN ICREP ICRIP ICRID ICNAME',
        f' 1    MZ {year:02d}280    10     0     1     1   -99   100    .5     0     0     0 -99',
        f'@C  ICBL  SH2O  SNH4  SNO3',
        f'{ic}',
        f'',
        f'*PLANTING DETAILS',
        f'@P PDATE EDATE  PPOP  PPOE  PLME  PLDS  PLRS  PLRD  PLDP  PLWT  PAGE  PENV  PLPH  SPRL                        PLNAME',
        f' 1 {year:02d}288   -99   3.7   3.7     S     R    61     0     7   -99   -99   -99   -99     0                        -99',
        f'',
        f'*FERTILIZERS (INORGANIC)',
        f'@F FDATE  FMCD  FACD  FDEP  FAMN  FAMP  FAMK  FAMC  FAMO  FOCD FERNAME',
        f'{fert}',
        f'',
        f'*SIMULATION CONTROLS',
        f'@N GENERAL     NYERS NREPS START SDATE RSEED SNAME.................... SMODEL',
        f' 1 GE          {nyers:>5}     1     S {year:02d}280  2150 KADOMA N RATES',
        f'@N OPTIONS     WATER NITRO SYMBI PHOSP POTAS DISES  CHEM  TILL   CO2',
        f' 1 OP              Y     Y     N     N     N     N     N     N     M',
        f'@N METHODS     WTHER INCON LIGHT EVAPO INFIL PHOTO HYDRO NSWIT MESOM MESEV MESOL',
        f' 1 ME              M     M     E     R     S     R     R     1     G     S     2',
        f'@N MANAGEMENT  PLANT IRRIG FERTI RESID HARVS',
        f' 1 MA              R     N     R     N     M',
        f'@N OUTPUTS     FNAME OVVEW SUMRY FROPT GROUT CAOUT WAOUT NIOUT MIOUT DIOUT VBOSE CHOUT OPOUT',
        f' 1 OU              N     N     Y     1     N     N     N     N     N     N     0     N     N',
        f'',
        f'@  AUTOMATIC MANAGEMENT',
        f'@N PLANTING    PFRST PLAST PH2OL PH2OU PH2OD PSTMX PSTMN',
        f' 1 PL          {year:02d}280 {year:02d}300    40   100    30    40    10',
        f'@N IRRIGATION  IMDEP ITHRL ITHRU IROFF IMETH IRAMT IREFF',
        f' 1 IR             30    50   100 GS000 IR001    10     1',
        f'@N NITROGEN    NMDEP NMTHR NAMNT NCODE NAOFF',
        f' 1 NI             30    50    25 FE001 GS000',
        f'@N RESIDUES    RIPCN RTIME RIDEP',
        f' 1 RE            100     1    20',
        f'@N HARVEST     HFRST HLAST HPCNP HPCNR',
        f' 1 HA              0 {year + 1:02d}280   100     0',
        f'',
    ]
    return chr(10).join(lines)

kx = kado_filex()
d = L.work_dir("m5_kadoma")
L.stage(d, "Soil/SOIL.SOL", *[f"Weather/KADO{y:02d}01.WTH" for y in range(77, 91)])
L.write_cultivar_variants(d, "IB0035", {"IB0098": {"P1": 260.0, "P2": 0.600, "P5": 840.0, "G2": 600.0, "G3": 6.00, "PHINT": 38.90}},
                          "MZCER048", names={"IB0098": "MH-16"})
print([l for l in (d / "MZCER048.CUL").read_text(encoding="latin-1").splitlines() if l.startswith("IB0098")])
(d / "KADO9901.SNX").write_text(kx, encoding="latin-1")
print(kx[:kx.index("*FIELDS")])
s, _, r = L.run(d, "KADO9901.SNX", outputs="summary")
print(f"{len(s)} seasons in {r.seconds:.0f} s")
kad = s[["TRNO", "SDAT", "PDAT", "MDAT", "HWAM", "NICM", "NUCM", "NLCM", "PRCM", "ETCM"]].apply(pd.to_numeric, errors="coerce")
kad["rate"] = kad.TRNO.map(dict(enumerate(rates, 1)))
kad["season"] = [f"{y}/{(y + 1) % 100:02d}" for y in (kad.PDAT // 1000).astype(int)]
kad.pivot(index="season", columns="rate", values="HWAM")

soil NO3 0.201 ppm in every layer gives 5 kg N/ha


['IB0098 MH-16                . IB0001 260.0 0.600 840.0 600.0 6.000 38.90 ']


*EXP.DETAILS: KADO7701SN KADOMA MAIZE N RATES (COURSE EXERCISE)

*GENERAL
@PEOPLE
DSSAT training exercise, built in Python
@SITE
Cotton Research Institute, Kadoma, Zimbabwe

*TREATMENTS                        -------------FACTOR LEVELS------------
@N R O C TNAME.................... CU FL SA IC MP MI MF MR MC MT ME MH SM
 1 1 0 0 UREA 0 KG N/HA             1  1  0  1  1  0  0  0  0  0  0  0  1
 2 1 0 0 UREA 20 KG N/HA            1  1  0  1  1  0  1  0  0  0  0  0  1
 3 1 0 0 UREA 40 KG N/HA            1  1  0  1  1  0  2  0  0  0  0  0  1
 4 1 0 0 UREA 60 KG N/HA            1  1  0  1  1  0  3  0  0  0  0  0  1
 5 1 0 0 UREA 80 KG N/HA            1  1  0  1  1  0  4  0  0  0  0  0  1
 6 1 0 0 UREA 100 KG N/HA           1  1  0  1  1  0  5  0  0  0  0  0  1
 7 1 0 0 UREA 120 KG N/HA           1  1  0  1  1  0  6  0  0  0  0  0  1
 8 1 0 0 UREA 140 KG N/HA           1  1  0  1  1  0  7  0  0  0  0  0  1
 9 1 0 0 UREA 160 KG N/HA           1  1  0  1  1  0  8  0  0  0  0  0  1
10 1 0 0 URE

169 seasons in 12 s


rate,0,20,40,60,80,100,120,140,160,180,200,220,240
season,,,,,,,,,,,,,
1977/78,782,1810,3432,4958,5314,5315,5324,5330,5339,5348,5354,5362,5366
1978/79,548,1312,2355,3504,4638,4897,4942,4920,4902,4879,4870,4873,4893
1979/80,548,1038,1567,2130,2562,3093,3775,4080,4141,4201,4213,4210,4215
1980/81,563,1075,1833,2667,3681,4366,5241,5262,5301,5366,5401,5419,5426
1981/82,666,1256,2303,3420,3666,3767,3701,3703,3710,3713,3717,3724,3729
1982/83,618,920,1328,2080,2450,2510,2320,2280,2282,2283,2270,2272,2268
1983/84,682,1570,2579,3321,3953,3872,3541,3402,3338,3300,3297,3297,3291
1984/85,673,1287,2280,3763,4569,4723,4840,4884,4901,4900,4895,4889,4880
1985/86,433,1234,2270,2544,2523,2536,2541,2545,2544,2548,2549,2544,2546


### The gross margin

The exercise's fictitious prices, in Zimbabwean dollars: `GM = Y x P - N x C - F`, with `Y` the yield in t/ha,
`P` = Z$ 400/t, `N` the amount of urea used and `C` = Z$ 2.0/kg, `F` = Z$ 750/ha of fixed costs. The exercise
calls `N` both "the amount of urea used" and "the amount of urea N"; the notebook prices each kilogram of N at
Z$ 2.0 and the page lets you change that (urea is 46% N, so Z$ 2.0 per kilogram of urea would be about Z$ 4.35
per kilogram of N).

In [7]:
P, C, F = 400.0, 2.0, 750.0
kad["GM"] = kad.HWAM / 1000 * P - kad.rate * C - F
best_season = kad.loc[kad.groupby("season").GM.idxmax(), ["season", "rate", "HWAM", "GM", "PRCM"]]
blanket = kad.groupby("rate").agg(mean_yield=("HWAM", "mean"), sd_yield=("HWAM", "std"), mean_GM=("GM", "mean"),
                                  sd_GM=("GM", "std"), mean_leached=("NLCM", "mean"), mean_uptake=("NUCM", "mean")).reset_index()
print("blanket optimum:", int(blanket.loc[blanket.mean_GM.idxmax(), "rate"]), "kg N/ha")
display(best_season)
kmon = None
from wth import read_wth
kw = pd.concat([read_wth(L.CSM_DATA / f"Weather/KADO{y:02d}01.WTH")[1] for y in range(77, 91)]).rename(columns=str.upper)
kmon = kw.assign(year=kw.index.year, month=kw.index.month).groupby(["year", "month"]).RAIN.sum().groupby("month").mean().reset_index()
L.export(kad[["season", "rate", "HWAM", "NUCM", "NLCM", "PRCM", "ETCM", "GM"]], M, "kadoma")
L.export(blanket, M, "kadoma_blanket"); L.export(best_season, M, "kadoma_best"); L.export(kmon, M, "kadoma_climate")
L.export(pd.DataFrame([{"P": P, "C": C, "F": F, "ppm_no3": ppm}]), M, "kadoma_setup", 4)
L.export(pd.DataFrame([{"section": "FileX", "text": kx[:kx.index("*FIELDS")].rstrip() + "\n..."}]), M, "kadoma_filex")
blanket.round(0)

blanket optimum: 120 kg N/ha


,season,rate,HWAM,GM,PRCM
52,1977/78,80,5314,1215.6,700
66,1978/79,100,4897,1008.8,439
93,1979/80,140,4080,602.0,613
81,1980/81,120,5241,1106.4,727
69,1981/82,100,3767,556.8,301
57,1982/83,80,2450,70.0,395
58,1983/84,80,3953,671.2,371
85,1984/85,120,4840,946.0,814
47,1985/86,60,2544,147.6,718
100,1986/87,140,2816,96.4,455


,rate,mean_yield,sd_yield,mean_GM,sd_GM,mean_leached,mean_uptake
0,0,599.0,122.0,-510.0,49.0,8.0,29.0
1,20,1212.0,304.0,-305.0,122.0,7.0,47.0
2,40,1940.0,675.0,-54.0,270.0,8.0,63.0
3,60,2653.0,1087.0,191.0,435.0,9.0,78.0
4,80,3125.0,1267.0,340.0,507.0,12.0,91.0
5,100,3269.0,1312.0,358.0,525.0,16.0,100.0
6,120,3397.0,1385.0,369.0,554.0,21.0,105.0
7,140,3410.0,1404.0,334.0,561.0,26.0,109.0
8,160,3409.0,1415.0,294.0,566.0,32.0,112.0
9,180,3409.0,1429.0,254.0,572.0,37.0,114.0


## 3. Regional risk analysis and adoption: millet in Niger

The course's `NIGE1101.SNX` simulates millet at Tillabery, Niger, for 10 years of measured weather, two
cultivars on two soils: a medium alfisol (`IBML910083`, soil type A) and a Niger sandy soil (`IBML910001`, soil
type B). The cultivars are entered in the cultivar file as the exercise gives them:

| | P1 | P2O | P2R | P5 | G1 | G4 | PHINT | GT | G5 |
|---|---|---|---|---|---|---|---|---|---|
| Niger 1 (`NI0001`) | 120.0 | 12.00 | 142.0 | 590.0 | 1.00 | 0.60 | 43.00 | 1.0 | 11.0 |
| Niger 2 (`NI0002`) | 120.0 | 12.00 | 125.0 | 360.0 | 2.00 | 0.50 | 43.00 | 1.0 | 11.0 |

The region: 100 farms on soil A (2 ha each on average; 20% of these farmers do not grow millet) and 150 farms on
soil B (1.8 ha each). Part B adds a heavy price penalty for late maturity, `price (CFA/t) = 300 - 0.02 D²` with
`D` the days from planting to maturity, and compares the cultivars by `utility = E - 0.35 V`, with `E` the mean
gross margin and `V` its variance.

In [8]:
d = L.work_dir("m5_niger")
L.stage_course(d, "NIGE1101.SNX")
L.stage(d, "Soil/SOIL.SOL", *[f"Weather/NITL{y:02d}01.WTH" for y in range(11, 22)])
NI = {"NI0001": {"P1": 120.0, "P2O": 12.00, "P2R": 142.0, "P5": 590.0, "G1": 1.00, "G4": 0.60, "PHINT": 43.00, "GT": 1.0, "G5": 11.0},
      "NI0002": {"P1": 120.0, "P2O": 12.00, "P2R": 125.0, "P5": 360.0, "G1": 2.00, "G4": 0.50, "PHINT": 43.00, "GT": 1.0, "G5": 11.0}}
L.write_cultivar_variants(d, "990001", NI, "MLCER048", names={"NI0001": "Niger 1", "NI0002": "Niger 2"})
print("\n".join(l for l in (d / "MLCER048.CUL").read_text(encoding="latin-1").splitlines() if l.startswith(("NI000", "990001"))))
s, _, r = L.run(d, "NIGE1101.SNX", outputs="summary")
print(f"{len(s)} seasons in {r.seconds:.0f} s")
ng = s[["TRNO", "PDAT", "MDAT", "HWAM", "CWAM", "PRCM", "ETCM"]].apply(pd.to_numeric, errors="coerce")
ng["soil"] = ng.TRNO.map({1: "A", 2: "A", 3: "B", 4: "B"})
ng["cultivar"] = ng.TRNO.map({1: "Niger 1", 2: "Niger 2", 3: "Niger 1", 4: "Niger 2"})
ng["year"] = (ng.PDAT // 1000).astype(int)
ng["D"] = [(L._date(m) - L._date(p)).days if pd.notna(m) and pd.notna(p) else np.nan for p, m in zip(ng.PDAT, ng.MDAT)]
ng["price"] = 300 - 0.02 * ng.D ** 2
ng["GM"] = ng.HWAM / 1000 * ng.price
summ_ng = ng.groupby(["soil", "cultivar"]).agg(mean_yield=("HWAM", "mean"), sd_yield=("HWAM", "std"), mean_D=("D", "mean"),
                                             mean_price=("price", "mean"), E=("GM", "mean"), V=("GM", "var")).reset_index()
summ_ng["utility"] = summ_ng.E - 0.35 * summ_ng.V
farms = {"A": 100 * 0.8 * 2.0, "B": 150 * 1.8}
print("millet area, ha:", farms)
summ_ng

990001 NORTH VARIETY        . IB0001 120.0 12.00 125.0 360.0  2.00  0.50 43.00  1.00  11.0
NI0001 Niger 1              . IB0001 120.0 12.00 142.0 590.0  1.00  0.60 43.00  1.00  11.0
NI0002 Niger 2              . IB0001 120.0 12.00 125.0 360.0  2.00  0.50 43.00  1.00  11.0


40 seasons in 3 s
millet area, ha: {'A': 160.0, 'B': 270.0}


,soil,cultivar,mean_yield,sd_yield,mean_D,mean_price,E,V,utility
0,A,Niger 1,2606.7,780.738689,104.4,81.768,214.298426,7246.129654,-2321.846953
1,A,Niger 2,1412.5,446.246880,88.0,144.912,204.194460,4031.378455,-1206.787999
2,B,Niger 1,1394.6,360.545636,105.4,77.508,109.060976,1856.411129,-540.682919
3,B,Niger 2,768.4,179.594729,89.0,141.308,107.854786,575.540154,-93.584268


In [9]:
L.export(ng[["year", "soil", "cultivar", "HWAM", "D", "price", "GM", "PRCM"]], M, "niger")
L.export(summ_ng, M, "niger_summary", 4)
L.export(pd.DataFrame([{"soil": k, "millet_ha": v} for k, v in farms.items()]), M, "niger_farms")

WindowsPath('C:/Users/benny/OneDrive/Documents/Github/xdssat/docs/learn/data/module5/niger_farms.csv')

## 4. Things to try

- Change `WTHER` of `UFGA7812.SNX` from `W` to `M` and use measured Gainesville weather files instead
  (dssat-csm-data has several years). Do the strategies keep their ranking?
- Exercise 1 asks what happens when each irrigation costs $25 instead of $0.50. The page has a slider for it.
- At Kadoma, plant a month later. Does the blanket N rate change?
- Exercise 4: set up a seasonal experiment of your own, 4 to 8 treatments, at least 8 years, with prices that
  fit the place.
- Do your own risk assessment: one crop, one decision, at least 20 seasons, and a statement of what you
  assumed.

In [10]:
info = pd.DataFrame([{"engine": str(L.config.engine_exe()), "csm_data_commit": L.CSM_DATA_COMMIT,
                      "csm_os_commit": L.CSM_OS_COMMIT, "run_on": time.strftime("%Y-%m-%d %H:%M")}])
L.export(info, M, "run_info")
info.T

,0
engine,C:\TEMP\dsn\dssat\dscsm048.exe
csm_data_commit,99c95b340acb9515c2d0a8ab582f5fbbf076f345
csm_os_commit,25c93b9a2f73f8ec45bf441b198dac5fb7b22849
run_on,2026-11-06 23:53
